In [12]:
import pandas as pd

train_df = pd.read_csv("../data/train.csv")
pred_df = pd.read_csv("../predictions-textmeta/train_predictions_xgboost_clf.csv")

In [13]:
analysis_df = pd.merge(train_df, pred_df, on="id", how="inner")
cols_to_drop = [col for col in analysis_df.columns if col.startswith("actual")]
print(cols_to_drop)

['actual_emotional_state', 'actual_intensity']


In [14]:
analysis_df = analysis_df.drop(columns=cols_to_drop)
#error masks
state_error_mask = (
    analysis_df["emotional_state"] != analysis_df["predicted_emotional_state"]
)
intensity_error_mask = analysis_df["intensity"] != analysis_df["predicted_intensity"]
state_uncertainty_mask = analysis_df["uncertainty_flag"] == True
intensity_uncertainty_mask = analysis_df["uncertainty_flag_intensity"] == True

problem_df = analysis_df[state_error_mask | intensity_error_mask | state_uncertainty_mask | intensity_uncertainty_mask]


In [20]:
print("Problematic cases:" + str(problem_df.shape[0]))
def flag_error_type(row):
    errors = []
    if row["emotional_state"] != row["predicted_emotional_state"]:
        errors.append("State_Wrong")
    if row["intensity"] != row["predicted_intensity"]:
        errors.append("Intensity_Wrong")
    for col in row.index:
        if 'uncertain' in col.lower() and row[col] == True:
            # Clean up the name for the label (e.g. 'intensity_uncertainty_flag' -> 'Intensity_Uncertain')
            label = col.replace('_flag', '').replace('_uncertainty', '_Uncertain').title()
            errors.append(label)
            
    return " & ".join(errors)


problem_df["error_type"] = problem_df.apply(flag_error_type, axis=1)

Problematic cases:503


In [21]:
print(problem_df["error_type"].value_counts())

error_type
Uncertainty_Intensity                                                  205
State_Wrong & Intensity_Wrong                                          103
Intensity_Wrong & Uncertainty_Intensity                                 45
State_Wrong & Intensity_Wrong & Uncertainty_Intensity                   41
State_Wrong                                                             33
Uncertainty & Uncertainty_Intensity                                     16
Intensity_Wrong                                                         16
Uncertainty                                                             13
State_Wrong & Uncertainty_Intensity                                      8
State_Wrong & Intensity_Wrong & Uncertainty & Uncertainty_Intensity      8
Intensity_Wrong & Uncertainty & Uncertainty_Intensity                    6
State_Wrong & Intensity_Wrong & Uncertainty                              3
State_Wrong & Uncertainty                                                3
State_Wrong & 